# E6 - Associative recall: bo loc suy tu corpus co lam mat kha nang tam xa khong?

Repo: https://github.com/KienNguyenDev2711/Hyena-Attention-Study

## Cau hoi

E0-b sinh ra bo loc co do dai hieu dung **trung vi 2,9 token**, voi **142/256 kenh nam trong 4 token**. Cau hinh do bien Hyena thanh gan nhu mot tich chap cuc bo.

No **co the giam perplexity** (vi thong tin trong van ban that su tap trung o khoang cach ngan) trong khi da **vut bo chinh kha nang tam xa** lam nen gia tri cua Hyena. Perplexity khong lo ra dieu do: mot mo hinh chi nhin 4 token van dat PPL kha tot tren van ban tu nhien.

Associative recall dat cau tra loi o xa va khong cho doan mo tu ngu canh cuc bo, nen no lo ra ngay diem yeu nay. Day cung la cong cu chan doan cua chinh paper (Table 5: Hyena 97,2% vs H3 0,6% o do dai 131072).

## Trang thai: CO MOT VAN DE CHUA GIAI QUYET

Test R5 (`tests/test_recall.py`) hien **THAT BAI**: attention 2 lop chi dat **0,128** so voi muc doan mo **0,100** o cau hinh vocab=10, L=33, 8 epoch.

**Khong duoc dung E6 de ket luan bat cu dieu gi cho toi khi R5 DAT.** Neu tac vu tu no khong giai duoc, thi Hyena dat 10% khong chung minh "Hyena mat kha nang tam xa" - no chi chung minh thuoc do bi hong.

Muc 3 duoi day quet cau hinh de tim diem ma attention that su giai duoc.

## Cai dat notebook

Settings ben phai: **Accelerator = GPU T4 x2**, **Internet = On**.

In [ ]:
REPO_URL = "https://github.com/KienNguyenDev2711/Hyena-Attention-Study.git"
WORK = "/kaggle/working/Hyena-Attention-Study"

import os, shutil, subprocess, sys

if os.path.isdir(WORK):
    shutil.rmtree(WORK)
subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, WORK], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

import torch
print("torch", torch.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} - {p.total_memory/2**30:.1f} GB")
else:
    print("\n" + "!" * 70)
    print("!! CHUA BAT GPU - Settings > Accelerator > GPU T4 x2")
    print("!! Test R5 se bi BO QUA va toan bo notebook nay vo nghia")
    print("!" * 70)

## 2. Kiem chung cai dat

Tren GPU, test R5 se tu dong chay (tren may khong co GPU no bi bo qua co chu dich, de khong nau CPU may ca nhan).

In [ ]:
!python tests/test_models.py
!python tests/test_pipeline.py
!python tests/test_morphology.py
!python tests/test_recall.py

## 3. Quet chan doan: cau hinh nao attention giai duoc?

Ba gia thuyet ve nguyen nhan R5 that bai:

| | Gia thuyet | Cach kiem |
|---|---|---|
| **H-a** | Thieu tin hieu huan luyen. Moi chuoi chi cho DUNG MOT nhan (vi tri cuoi), nen 6000 mau = 6000 tin hieu. Co che induction head can nhieu buoc hon nhieu. | tang epoch va n_train |
| **H-b** | Chuoi/vocab qua kho so voi ngan sach. | giam vocab va do dai |
| **H-c** | Can nhieu hon 2 lop. | thu 3-4 lop |

Tieu chi dat: **do chinh xac > muc doan mo + 0,30**.

In [ ]:
import time

import numpy as np
import torch
import torch.nn.functional as F

from hyena_study.data.synthetic import RecallConfig, build_recall_dataset, chance_accuracy
from hyena_study.models import LMConfig, SequenceLM

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def trial(vocab, seq_len, layers, epochs, n_train, lr, d_model=64, seed=0, alpha=None):
    torch.manual_seed(seed); np.random.seed(seed)
    cfg = RecallConfig(vocab_size=vocab, seq_len=seq_len, n_train=n_train,
                       n_val=10, n_test=2000, seed=0)
    d = build_recall_dataset(cfg)
    xtr, ytr = d["train"]; xte, yte = d["test"]
    L = xtr.shape[1]

    from hyena_study.models import HyenaFilterConfig
    m = SequenceLM(LMConfig(
        vocab_size=vocab, d_model=d_model, layer_spec=layers, max_seq_len=L,
        dropout=0.0, n_heads=4,
        hyena_filter=HyenaFilterConfig(alpha_values=alpha),
    )).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=lr)

    xtr_t = torch.from_numpy(xtr).to(DEV); ytr_t = torch.from_numpy(ytr).to(DEV)
    xte_t = torch.from_numpy(xte).to(DEV)
    bs, steps, t0 = 64, 0, time.time()
    total = epochs * (len(xtr) // bs)
    for _ in range(epochs):
        perm = torch.randperm(len(xtr), device=DEV)
        for i in range(0, len(xtr) - bs + 1, bs):
            idx = perm[i:i + bs]
            for g in opt.param_groups:
                g["lr"] = lr * min(1.0, (steps + 1) / max(total * 0.05, 1))
            loss = F.cross_entropy(m(xtr_t[idx])[:, -1, :], ytr_t[idx])
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
            steps += 1
    m.eval()
    with torch.no_grad():
        pred = m(xte_t)[:, -1, :].argmax(-1).cpu().numpy()
    return float((pred == yte).mean()), chance_accuracy(cfg), steps, time.time() - t0


GRID = [
    # vocab, seq_len, layers, epochs, n_train, lr   -- ghi chu
    (10, 33, "AA",  8,   6000, 3e-3),   # cau hinh dang FAIL, de doi chieu
    (10, 33, "AA",  60, 20000, 1e-3),   # H-a: nhieu buoc + nhieu du lieu
    (10, 33, "AAA", 60, 20000, 1e-3),   # H-c: 3 lop
    (5,  17, "AA",  60, 20000, 1e-3),   # H-b: de hon han
    (20, 65, "AAA", 60, 40000, 1e-3),   # muc tieu that su muon dung
]

print(f"{'vocab':>6}{'L':>5}{'lop':>6}{'buoc':>8}{'acc':>8}{'doan mo':>9}{'giay':>7}  ket luan")
print("-" * 62)
solvable = []
for vocab, sl, layers, ep, ntr, lr in GRID:
    acc, ch, st, el = trial(vocab, sl, layers, ep, ntr, lr)
    ok = acc > ch + 0.30
    if ok:
        solvable.append((vocab, sl, layers, ep, ntr, lr, acc))
    print(f"{vocab:>6}{sl:>5}{layers:>6}{st:>8}{acc:>8.3f}{ch:>9.3f}{el:>7.0f}"
          + ("  GIAI DUOC" if ok else "  chua dat"))

print()
if solvable:
    print(f"Co {len(solvable)} cau hinh giai duoc. Dung cau hinh KHO NHAT trong so do cho E6,")
    print("vi cang kho thi cang de lo ra su khac biet giua cac toan tu.")
else:
    print("KHONG cau hinh nao dat. DUNG LAI - khong chay muc 4.")
    print("Tac vu hoac cach huan luyen dang co van de, phai sua truoc.")
    print("Huong tiep theo: tang manh epoch, hoac cap nhan o MOI vi tri gia tri")
    print("thay vi chi vi tri cuoi (tang tin hieu giam sat len ~n_pairs lan).")

## 4. E6 chinh - chi chay khi muc 3 tim duoc cau hinh giai duoc

So sanh bon cau hinh o **cung tac vu, cung ngan sach huan luyen**:

| Cau hinh | Y nghia |
|---|---|
| `AAAA` | Attention - moc tren, truy cap truc tiep moi vi tri |
| `HHHH` + `uniform` | Hyena voi khoang alpha nhom tu chon (doi chung 1) |
| `HHHH` + `logspace` | Hyena voi do dai hieu dung trai deu theo log (doi chung 2, cong bang) |
| `HHHH` + `corpus` | Hyena voi alpha suy tu corpus (de xuat) - trung vi 2,9 token |

**Du doan can kiem chung:** cau hinh `corpus` co do dai hieu dung rat ngan nen se **kem nhat** o tac vu nay, ke ca khi no tot nhat ve perplexity. Neu dieu do xay ra, do la mot phat hien co gia tri chu khong phai that bai - no cho thay khoi tao dua tren du lieu **danh doi kha nang tam xa lay chat luong cuc bo**.

In [ ]:
import json

import pandas as pd

from hyena_study.morphology import alphas_from_mi, logspaced_alphas

assert solvable, "Muc 3 chua tim duoc cau hinh giai duoc - khong chay tiep"
VOCAB, SEQ_LEN, LAYERS_A, EPOCHS, N_TRAIN, LR, _ = solvable[-1]
D_MODEL, N_LAYERS = 64, len(LAYERS_A)
L_REAL = 2 * ((SEQ_LEN - 1) // 2) + 1
print(f"Dung cau hinh: vocab={VOCAB} L={L_REAL} {N_LAYERS} lop "
      f"epochs={EPOCHS} n_train={N_TRAIN}\n")

# Alpha phai duoc SINH LAI cho dung d_model va seq_len cua tac vu nay.
# Dung lai file alpha sinh cho d_model=256 / L=512 se sai do dai hieu dung.
mi = pd.read_csv("results/E0b_mi_decay_vi_bpe_k500.csv")
corpus_alpha = alphas_from_mi(mi["lag"].values, mi["mi_corrected_nats"].values,
                              d_model=D_MODEL, seq_len=L_REAL).alpha
logspace_alpha = logspaced_alphas(D_MODEL, seq_len=L_REAL).alpha

import numpy as np
for nm, a in (("corpus", corpus_alpha), ("logspace", logspace_alpha)):
    e = L_REAL / np.array(a)
    print(f"  {nm:<9} do dai hieu dung: trung vi {np.median(e):.1f} token, "
          f"{(e <= 4).sum()}/{D_MODEL} kenh <= 4 token")

CONFIGS = [
    ("attention",       "A" * N_LAYERS, None),
    ("hyena_uniform",   "H" * N_LAYERS, None),
    ("hyena_logspace",  "H" * N_LAYERS, logspace_alpha),
    ("hyena_corpus",    "H" * N_LAYERS, corpus_alpha),
]
SEEDS = [0, 1]

rows = []
for name, spec, alpha in CONFIGS:
    for sd in SEEDS:
        acc, ch, st, el = trial(VOCAB, SEQ_LEN, spec, EPOCHS, N_TRAIN, LR,
                                d_model=D_MODEL, seed=sd, alpha=alpha)
        rows.append({"config": name, "seed": sd, "accuracy": acc,
                     "chance": ch, "seconds": el})
        print(f"  {name:<16} seed {sd}: acc {acc:.4f} (doan mo {ch:.3f}) [{el:.0f}s]")

df = pd.DataFrame(rows)
summary = df.groupby("config")["accuracy"].agg(["mean", "std", "min", "max"])
print("\n" + "=" * 60)
print(summary.to_string())
print(f"\nMuc doan mo: {rows[0]['chance']:.4f}")

df.to_csv("results/E6_recall_comparison.csv", index=False)
print("\nDa ghi results/E6_recall_comparison.csv")
print("\nDoc ket qua:")
print("  - corpus THAP hon logspace  => khoi tao tu du lieu danh doi kha nang tam xa")
print("  - corpus NGANG logspace     => bo loc ngan khong gay hai o do dai nay")
print("  - moi cau hinh Hyena deu ~ doan mo => Hyena khong giai duoc tac vu nay o")
print("    quy mo nay; phai bao cao trung thuc, KHONG duoc quy cho rieng bo loc corpus")

In [ ]:
import shutil
from pathlib import Path

out = Path("/kaggle/working/E6_ketqua")
if out.exists():
    shutil.rmtree(out)
out.mkdir(parents=True)
for f in Path("results").glob("E6*"):
    shutil.copy(f, out / f.name)
shutil.make_archive("/kaggle/working/E6_ketqua", "zip", out)
print("Da gom:", sorted(p.name for p in out.iterdir()))
print("Tai /kaggle/working/E6_ketqua.zip o panel Output ben phai.")